##  Life Insurance Policy — Data Quality Expectation Suite
|Number of Policices|Covered_Lives| Columns|
|---|---|---|
|2000|8,359|48|

| Dimension | Expectations |
|---|---|
| Completeness | Null checks on critical columns |
| Validity | Ranges, formats, allowed values |
| Consistency | Cross-column business rules |
| Freshness | Timeliness of date fields |

In [1]:
import pandas as pd
import great_expectations as gx
from datetime import datetime
import warnings, os
warnings.filterwarnings('ignore')

print(f'✅ Great Expectations version: {gx.__version__}')

✅ Great Expectations version: 1.18.0


In [2]:
# Import sample data into Pandas DataFrame.
df = pd.read_csv("insurance_policy_covered_lives_raw_dataset.csv")

# Convert date columns
date_cols = ['DOB', 'Issue_Date', 'Last_Paid_Date', 'Date_of_Purchase',
             'Policy_Anniversary_Date', 'Maturity_Date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

today = pd.Timestamp(datetime.today().date())

print(f'✅ Loaded: {len(df):,} rows × {len(df.columns)} columns')
df.head(3)

✅ Loaded: 8,359 rows × 48 columns


,Policy_Number,Customer_ID,Covered_Life_ID,Product_ID,Product_Name,Insured_Class,Role,DOB,Gender,Issue_Date,...,Initial_Premium,Current_Premium,Update_Percentage,Total_Premium_Paid,Total_Premium_Due,No_of_Outstanding_Premium,Total_Outstanding_Premium,Total_Refund_Due,Maturity_Date,Term
0,CLP550527159,CUST571126,UCF550527159-CL05,CLP006,Classic Legacy Plan,PARENT-IN-LAW,Life Assured,1967-02-13,F,2024-08-14,...,519.30,571.23,10.0,10282.14,11995.83,3,1713.69,0.0,2054-08-15,360
1,ESP605706734,CUST509779,UES605706734-CL05,ESP005,Ultimate Educational Support Plan,CHILD,Beneficiary,2005-07-12,M,2022-08-11,...,617.53,586.65,-5.0,24639.30,24639.30,0,0.00,0.0,2052-08-11,360
2,CLP387329239,CUST849079,UCF387329239-CL05,CLP006,Classic Legacy Plan,PARENT,Life Assured,1958-02-20,M,2023-01-06,...,291.76,320.94,10.0,11232.90,11232.90,0,0.00,0.0,2048-06-01,300


## Initialize GE File Context

In [3]:
# Complete Data Profiling Summary
Profile = pd.DataFrame({
    'Rows':df.shape[0],
    'Columns':df.shape[1],
    'Data Type':df.dtypes.astype(str),
    'Missing Values':df.isnull().sum(),
    'Missing %':round(df.isnull().mean()*100,2),
    'Unique Values':df.nunique()
})
display(Profile)

,Rows,Columns,Data Type,Missing Values,Missing %,Unique Values
Policy_Number,8359,48,object,0,0.00,1978
Customer_ID,8359,48,object,0,0.00,2000
Covered_Life_ID,8359,48,object,0,0.00,8276
Product_ID,8359,48,object,0,0.00,7
Product_Name,8359,48,object,0,0.00,21
Insured_Class,8359,48,object,0,0.00,6
Role,8359,48,object,0,0.00,2
DOB,8359,48,datetime64[ns],0,0.00,5707
Gender,8359,48,object,0,0.00,2
Issue_Date,8359,48,datetime64[ns],0,0.00,1466


In [6]:
# Creates great_expectations/ folder in your project root
context = gx.get_context(mode='file', project_root_dir='.')

# Connect dataframe as a datasource
data_source = context.data_sources.add_pandas('insurance_policy_data_set')
data_asset  = data_source.add_dataframe_asset('covered_lives_asset')
batch_def   = data_asset.add_batch_definition_whole_dataframe('full_batch')
batch       = batch_def.get_batch(batch_parameters={'dataframe': df})

print('✅ GE file context initialized')
print(f'   Data Docs will be saved to: ./great_expectations/uncommitted/data_docs/')

# Create Expectation.
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column = "Premium_Amount", min_value = 0, severity = "warning"
)

# Validate Batch using Expectation.
test = batch.validate(expectation)
print(f'✅ Connected — {len(df):,} rows loaded' if test.success else '❌ Connection failed')
#print(test)

✅ GE file context initialized
   Data Docs will be saved to: ./great_expectations/uncommitted/data_docs/


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

❌ Connection failed


## Build Expectation Suite

In [7]:
SUITE_NAME = 'insurance_covered_lives_dq_suite'

# Remove existing suite if re-running
try:
    context.suites.delete(SUITE_NAME)
except:
    pass

suite = context.suites.add(gx.ExpectationSuite(name=SUITE_NAME))
print(f'✅ Suite created: {SUITE_NAME}')

✅ Suite created: insurance_covered_lives_dq_suite


### Completeness — Null Checks

In [8]:
critical_cols = [
    'Policy_Number', 'Customer_ID', 'Covered_Life_ID',
    'Product_ID', 'Product_Name', 'Role', 'DOB', 'Gender',
    'Issue_Date', 'Date_of_Purchase', 'Premium_Amount',
    'Annual_Premium', 'Policy_Status', 'Contract_Status',
    'Payment_Frequency', 'Term', 'Maturity_Date', 'Sum_Assured'
]

for col in critical_cols:
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(
            column=col,
            meta={'dimension': 'Completeness'}
        )
    )

print(f'✅ Completeness: {len(critical_cols)} expectations added')

✅ Completeness: 18 expectations added


### Validity — Ranges, Categories, Formats


In [9]:
validity_expectations = [
    # Categorical
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='Gender',
        value_set=['M', 'F', 'Male', 'Female'],
        meta={'dimension': 'Validity'}
    ),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='Policy_Status',
        value_set=['ACTIVE', 'LAPSE', 'NEVER LAPSE', 'SURRENDERED', 'MATURED'],
        meta={'dimension': 'Validity', 'note': 'ACTIVEE and LAPSED are known data entry errors'}
    ),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='Payment_Frequency',
        value_set=['Monthly', 'Quarterly', 'Semi-Annual', 'Annual', 'Single'],
        meta={'dimension': 'Validity'}
    ),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='Loan_Eligible',
        value_set=['Y', 'N', 'Yes', 'No'],
        meta={'dimension': 'Validity'}
    ),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='Role',
        value_set=['Primary', 'Spouse', 'Child', 'Dependent', 'Nominee', 'Proposer'],
        meta={'dimension': 'Validity'}
    ),
    # Numeric ranges
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='Premium_Amount', min_value=0.01,
        meta={'dimension': 'Validity'}
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='Annual_Premium', min_value=0.01,
        meta={'dimension': 'Validity'}
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='Monthly_Premium', min_value=0.01,
        meta={'dimension': 'Validity'}
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='Sum_Assured', min_value=1,
        meta={'dimension': 'Validity'}
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='Term', min_value=12, max_value=480,
        meta={'dimension': 'Validity', 'note': 'Term is in months'}
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='Tenure_Years', min_value=0,
        meta={'dimension': 'Validity'}
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='No_of_Outstanding_Premium', min_value=0,
        meta={'dimension': 'Validity'}
    ),
    # Uniqueness
    gx.expectations.ExpectColumnValuesToBeUnique(
        column='Covered_Life_ID',
        meta={'dimension': 'Validity'}
    ),
    # DOB must be in the past
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='DOB',
        min_value='1900-01-01',
        max_value=str(today.date()),
        meta={'dimension': 'Validity'}
    ),
]

for exp in validity_expectations:
    suite.add_expectation(exp)

print(f'✅ Validity: {len(validity_expectations)} expectations added')

✅ Validity: 14 expectations added


### Consistency — Cross-Column Business Rules


In [10]:
consistency_results = []

def cross_check(label, mask, df_subset=None):
    if df_subset is None:
        df_subset = df
    violations = df_subset[mask]
    status     = '✅ PASS' if len(violations) == 0 else '❌ FAIL'
    pct        = round(len(violations) / len(df_subset) * 100, 2)
    consistency_results.append({
        'Dimension': 'Consistency',
        'Check'    : label,
        'Status'   : status,
        'Issues'   : len(violations),
        'Issue %'  : f'{pct}%'
    })
    print(f'{status} | {label:<52} | Issues: {len(violations):,} ({pct}%)')
    return violations

print(f"{'STATUS':<8} | {'CHECK':<52} | ISSUES")
print('-' * 80)

# Rule 1: Maturity_Date must be after Issue_Date
cross_check('Maturity_Date > Issue_Date',
    df['Maturity_Date'] <= df['Issue_Date'])

# Rule 2: Last_Paid_Date must not be before Issue_Date
cross_check('Last_Paid_Date >= Issue_Date',
    df['Last_Paid_Date'] < df['Issue_Date'])

# Rule 3: Date_of_Purchase must be on or after Issue_Date
cross_check('Date_of_Purchase >= Issue_Date',
    df['Date_of_Purchase'] < df['Issue_Date'])

# Rule 4: Annual_Premium >= Monthly_Premium * 12 (5% tolerance)
cross_check('Annual_Premium >= Monthly_Premium * 12',
    df['Annual_Premium'] < (df['Monthly_Premium'] * 12 * 0.95))

# Rule 5: Total_Premium_Paid <= Total_Premium_Due + Total_Refund_Due
cross_check('Total_Premium_Paid <= Total_Premium_Due + Refund',
    df['Total_Premium_Paid'] > (df['Total_Premium_Due'] + df['Total_Refund_Due']))

# Rule 6: Loan_Amount_Allowed = 0 when not loan eligible
cross_check('Loan_Amount_Allowed = 0 when not eligible',
    df['Loan_Eligible'].isin(['N', 'No']) & (df['Loan_Amount_Allowed'] > 0))

# Rule 7: Tenure_Years must not exceed Term (Term is in months)
cross_check('Tenure_Years <= Term (months)',
    df['Tenure_Years'] > (df['Term'] / 12))

# Rule 8: Maturity_Date ≈ Issue_Date + Term months (45-day tolerance)
expected = df['Issue_Date'] + pd.to_timedelta(df['Term'] * 30.44, unit='D')
gap      = (df['Maturity_Date'] - expected).abs()
mask_r8  = gap > pd.Timedelta(days=45)
status   = '✅ PASS' if not mask_r8.any() else '❌ FAIL'
issues   = int(mask_r8.sum())
pct      = round(issues / len(df) * 100, 2)
consistency_results.append({
    'Dimension': 'Consistency',
    'Check'    : 'Maturity_Date ≈ Issue_Date + Term (months)',
    'Status'   : status,
    'Issues'   : issues,
    'Issue %'  : f'{pct}%'
})
print(f'{status} | {"Maturity_Date ≈ Issue_Date + Term (months)":<52} | Issues: {issues:,} ({pct}%)')

print(f'\n✅ Consistency: {len(consistency_results)} rules checked')

STATUS   | CHECK                                                | ISSUES
--------------------------------------------------------------------------------
❌ FAIL | Maturity_Date > Issue_Date                           | Issues: 61 (0.73%)
❌ FAIL | Last_Paid_Date >= Issue_Date                         | Issues: 31 (0.37%)
❌ FAIL | Date_of_Purchase >= Issue_Date                       | Issues: 1,030 (12.32%)
✅ PASS | Annual_Premium >= Monthly_Premium * 12               | Issues: 0 (0.0%)
✅ PASS | Total_Premium_Paid <= Total_Premium_Due + Refund     | Issues: 0 (0.0%)
✅ PASS | Loan_Amount_Allowed = 0 when not eligible            | Issues: 0 (0.0%)
❌ FAIL | Tenure_Years <= Term (months)                        | Issues: 3,508 (41.97%)
❌ FAIL | Maturity_Date ≈ Issue_Date + Term (months)           | Issues: 1,071 (12.81%)

✅ Consistency: 8 rules checked


### Freshness — Timeliness Checks

In [11]:
freshness_results = []

def freshness_check(label, date_col, threshold_days):
    days_since = (today - df[date_col]).dt.days
    stale      = df[days_since > threshold_days]
    pct        = round(len(stale) / len(df) * 100, 2)
    fresh_pct  = round(100 - pct, 2)
    status     = '✅ PASS' if fresh_pct >= 80 else '❌ FAIL'
    freshness_results.append({
        'Dimension': 'Freshness',
        'Check'    : label,
        'Status'   : status,
        'Issues'   : len(stale),
        'Issue %'  : f'{pct}%'
    })
    print(f'{status} | {label:<52} | Stale: {len(stale):,} ({pct}%) | Fresh: {fresh_pct}%')

print(f"{'STATUS':<8} | {'CHECK':<52} | RESULT")
print('-' * 90)

freshness_check('Last_Paid_Date within 365 days',       'Last_Paid_Date',          365)
freshness_check('Policy_Anniversary_Date within 400d',  'Policy_Anniversary_Date', 400)
freshness_check('Date_of_Purchase not future-dated',    'Date_of_Purchase',        0)

# Future Issue_Date check
future_issue = df[df['Issue_Date'] > today]
status = '✅ PASS' if len(future_issue) == 0 else '❌ FAIL'
pct    = round(len(future_issue) / len(df) * 100, 2)
freshness_results.append({
    'Dimension': 'Freshness',
    'Check'    : 'Issue_Date not in future',
    'Status'   : status,
    'Issues'   : len(future_issue),
    'Issue %'  : f'{pct}%'
})
print(f'{status} | {"Issue_Date not in future":<52} | Issues: {len(future_issue):,}')

print(f'\n✅ Freshness: {len(freshness_results)} checks complete')

STATUS   | CHECK                                                | RESULT
------------------------------------------------------------------------------------------
✅ PASS | Last_Paid_Date within 365 days                       | Stale: 1,533 (18.34%) | Fresh: 81.66%
✅ PASS | Policy_Anniversary_Date within 400d                  | Stale: 1,640 (19.62%) | Fresh: 80.38%
❌ FAIL | Date_of_Purchase not future-dated                    | Stale: 2,125 (25.42%) | Fresh: 74.58%
✅ PASS | Issue_Date not in future                             | Issues: 0

✅ Freshness: 4 checks complete


## Run GE Validation & Generate Data Docs

In [13]:
VALIDATION_NAME   = 'insurance_validation'
CHECKPOINT_NAME   = 'insurance_policy_checkpoint'

# Clean up existing definitions if re-running
for name in [VALIDATION_NAME]:
    try:
        context.validation_definitions.delete(name)
    except:
        pass

for name in [CHECKPOINT_NAME]:
    try:
        context.checkpoints.delete(name)
    except:
        pass

# Create validation definition
validation_def = context.validation_definitions.add(
    gx.ValidationDefinition(
        name  = VALIDATION_NAME,
        data  = batch_def,
        suite = suite
    )
)

# Create and run checkpoint
checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name                    = CHECKPOINT_NAME,
        validation_definitions  = [validation_def]
    )
)

print('Running validation checkpoint...')
checkpoint_result = checkpoint.run(batch_parameters={'dataframe': df})

overall_success = checkpoint_result.success
print(f'\n{"✅ Checkpoint PASSED" if overall_success else "❌ Checkpoint FAILED — issues found (expected)"}')

# Build Data Docs
print('\n Building Data Docs...')
context.build_data_docs()
print(' Data Docs built!')
print('\n Open your report at:')
print('   ./great_expectations/uncommitted/data_docs/local_site/index.html')

Running validation checkpoint...


Calculating Metrics:   0%|          | 0/182 [00:00<?, ?it/s]


❌ Checkpoint FAILED — issues found (expected)

 Building Data Docs...
 Data Docs built!

 Open your report at:
   ./great_expectations/uncommitted/data_docs/local_site/index.html


## Executive Scorecard
Combines GE results with manual consistency and freshness checks.

In [14]:
import pandas as pd

# ── Parse GE validation results ───────────────────────────────────
gx_results = []
for vr in checkpoint_result.run_results.values():
    for result in vr.results:
        dim = result.expectation_config.meta.get('dimension', 'General')
        gx_results.append({
            'Dimension': dim,
            'Check'    : result.expectation_config.type,
            'Status'   : '✅ PASS' if result.success else '❌ FAIL',
            'Issues'   : result.result.get('unexpected_count', 0),
            'Issue %'  : f"{round(result.result.get('unexpected_percent', 0), 2)}%"
        })

# ── Combine all results ───────────────────────────────────────────
all_results = pd.DataFrame(gx_results + consistency_results + freshness_results)

def dim_score(dimension):
    subset = all_results[all_results['Dimension'] == dimension]
    if len(subset) == 0: return 0
    return round(subset['Status'].str.contains('PASS').sum() / len(subset) * 100, 1)

def grade(s):
    return '🟢 GOOD' if s >= 90 else ('🟡 FAIR' if s >= 70 else '🔴 POOR')

comp    = dim_score('Completeness')
val     = dim_score('Validity')
con     = dim_score('Consistency')
fresh   = dim_score('Freshness')
overall = round((comp + val + con + fresh) / 4, 1)

print('=' * 62)
print('   INSURANCE COVERED LIVES — DQ EXECUTIVE SCORECARD')
print('=' * 62)
print(f'   Dataset      : Insurance Covered Lives')
print(f'   Records      : {len(df):,}')
print(f'   Columns      : {len(df.columns)}')
#print(f'   Report Date  : {today.strftime("%B %d, %Y")}')
print('=' * 62)
print(f'   DIMENSION       CHECKS   SCORE    GRADE')
print(f'   {"-"*52}')
for dim, score in [('Completeness', comp), ('Validity', val),
                   ('Consistency', con),   ('Freshness', fresh)]:
    n = len(all_results[all_results['Dimension'] == dim])
    print(f'   {dim:<16} {n:>4}     {score:>5.1f}%   {grade(score)}')
print(f'   {"-"*52}')
print(f'   OVERALL DQ SCORE          {overall:>5.1f}%   {grade(overall)}')
print('=' * 70)

failures = all_results[all_results['Status'].str.contains('FAIL')]
if len(failures) > 0:
    print(f'\n   FAILED CHECKS ({len(failures)}):')
    for _, row in failures.iterrows():
        print(f'   • [{row["Dimension"]}] {row["Check"]} — {row["Issues"]} issues ({row["Issue %"]})')

print('=' * 70)

   INSURANCE COVERED LIVES — DQ EXECUTIVE SCORECARD
   Dataset      : Insurance Covered Lives
   Records      : 8,359
   Columns      : 48
   DIMENSION       CHECKS   SCORE    GRADE
   ----------------------------------------------------
   Completeness       18      88.9%   🟡 FAIR
   Validity           14      64.3%   🔴 POOR
   Consistency         8      37.5%   🔴 POOR
   Freshness           4      75.0%   🟡 FAIR
   ----------------------------------------------------
   OVERALL DQ SCORE           66.4%   🔴 POOR

   FAILED CHECKS (13):
   • [Validity] expect_column_values_to_be_between — 0 issues (0%)
   • [Validity] expect_column_values_to_be_unique — 166 issues (1.99%)
   • [Validity] expect_column_values_to_be_in_set — 8359 issues (100.0%)
   • [Completeness] expect_column_values_to_not_be_null — 6234 issues (74.58%)
   • [Validity] expect_column_values_to_be_between — 83 issues (0.99%)
   • [Completeness] expect_column_values_to_not_be_null — 12 issues (0.14%)
   • [Validity] expe

## 7. Open Data Docs in Browser

In [15]:
# Opens the Data Docs HTML report in your default browser
context.open_data_docs()
print('✅ Data Docs opened in browser')

✅ Data Docs opened in browser


In [16]:
!pip freeze > requirements.txt